In [6]:
# =========================
# Reproducible XAI Script (SHAP + LIME)
# - FIT TO TREE: 9_best_model_no_weight/
# - GES ONLY
# - STRICT: XAI features MUST equal previously used features (features_used.csv -> FEATURES_JSON)
# - F : inferred from results filename -> 34 edges
# - OF: inferred from results filename -> 47 features (K_EDGE = 47 - n_original)
# =========================

import os
import re
import json
import time
import hashlib
import random
import warnings
from typing import Dict, Any, List

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import shap
from lime.lime_tabular import LimeTabularExplainer

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings("ignore")


# ============================================================
# 0) Paths (FIT TO YOUR TREE)
# ============================================================
RANDOM_STATE = 42
SAVE_METADATA: bool = False

# results tables may not have MODEL column -> fallback
DEFAULT_MODEL_NAME = "LightGBM"  # or "XGBoost"

BASE_DIR = os.getcwd()
print("[CWD]", BASE_DIR)

DATA_DIR    = os.path.join(BASE_DIR, "add_feature_dataset", "GES")
RESULTS_DIR = os.path.join(BASE_DIR, "results_tables")

DATA_TRAIN = os.path.join(DATA_DIR, "data_with_features_GES_train.csv")
DATA_VAL   = os.path.join(DATA_DIR, "data_with_features_GES_val.csv")
DATA_TEST  = os.path.join(DATA_DIR, "data_with_features_GES_test.csv")

FEATURE_LIST_PATH  = os.path.join(DATA_DIR, "feature_list.csv")
SPLIT_INDICES_PATH = os.path.join(DATA_DIR, "split_indices.json")  # optional

FEATURES_USED_PATH = os.path.join(BASE_DIR, "features_used.csv")

RESULTS_F_MEAN_PATH  = os.path.join(RESULTS_DIR, "results_F_GES_34edges_5seeds_mean.csv")
RESULTS_F_RUNS_PATH  = os.path.join(RESULTS_DIR, "results_F_GES_34edges_5seeds_runs.csv")
RESULTS_OF_MEAN_PATH = os.path.join(RESULTS_DIR, "results_OF_GES_47features_5seeds_mean.csv")
RESULTS_OF_RUNS_PATH = os.path.join(RESULTS_DIR, "results_OF_GES_47features_5seeds_runs.csv")

XAI_OUT_BASE = os.path.join(BASE_DIR, "xai_outputs_repro")
os.makedirs(XAI_OUT_BASE, exist_ok=True)

# sanity check
for p in [
    DATA_TRAIN, DATA_VAL, DATA_TEST,
    FEATURE_LIST_PATH,
    FEATURES_USED_PATH,
    RESULTS_F_MEAN_PATH, RESULTS_F_RUNS_PATH,
    RESULTS_OF_MEAN_PATH, RESULTS_OF_RUNS_PATH,
]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"[MISSING] {p}")

print("[OK] all paths exist")


# ============================================================
# 1) Determinism utilities
# ============================================================
UINT32_MAX = 2**32 - 1

def seed_uint32(x: int) -> int:
    return int(x) % UINT32_MAX

def set_seed_everywhere(seed: int):
    s = seed_uint32(seed)
    random.seed(s)
    np.random.seed(s)

set_seed_everywhere(RANDOM_STATE)


# ============================================================
# 2) HARD BLOCK: index-like remover
# ============================================================
def is_index_col_name(col: str) -> bool:
    c = str(col).strip()
    cl = c.lower()
    if c.startswith("Unnamed") or cl.startswith("unnamed"):
        return True
    if cl in {"index", "_index"}:
        return True
    if cl.endswith("_index"):
        return True
    return False

def looks_like_index_series(s: pd.Series) -> bool:
    try:
        v = pd.to_numeric(s, errors="coerce")
        if v.isna().mean() > 0.3:
            return False
        n = len(v)
        if n <= 5:
            return False
        uniq_ratio = v.nunique(dropna=True) / float(n)
        if uniq_ratio < 0.98:
            return False
        vv = v.to_numpy()
        if np.all(vv == np.arange(n)):
            return True
        if np.all(vv == np.arange(1, n + 1)):
            return True
        if np.all(np.diff(vv) >= 0):
            dif = np.diff(vv)
            if np.mean(np.abs(dif - 1.0) < 1e-9) > 0.95:
                return True
    except Exception:
        return False
    return False

def drop_all_indexlike_cols(df: pd.DataFrame, name: str, aggressive_value_check: bool = True) -> pd.DataFrame:
    if df is None or df.shape[1] == 0:
        return df

    drop_cols = []
    for c in list(df.columns):
        if is_index_col_name(c):
            drop_cols.append(c)

    if aggressive_value_check:
        for c in list(df.columns):
            if c in drop_cols:
                continue
            try:
                if looks_like_index_series(df[c]):
                    drop_cols.append(c)
            except Exception:
                pass

    if drop_cols:
        drop_cols = list(dict.fromkeys(drop_cols))
        print(f"[DROP] {name}: {drop_cols}")
        df = df.drop(columns=drop_cols)

    bad = [c for c in df.columns if str(c).lower().startswith("unnamed")]
    if bad:
        raise RuntimeError(f"[FATAL] {name}: Unnamed columns still exist after drop: {bad}")
    return df

def sanitize_feature_list(cols: List[str]) -> List[str]:
    cols2 = [c for c in cols if not is_index_col_name(c) and not str(c).lower().startswith("unnamed")]
    seen = set()
    out = []
    for c in cols2:
        if c not in seen:
            out.append(c)
            seen.add(c)
    if any(str(c).lower().startswith("unnamed") for c in out):
        raise RuntimeError("[FATAL] sanitize_feature_list produced Unnamed columns.")
    return out

def read_csv_safely(path: str, name: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df = drop_all_indexlike_cols(df, name=name, aggressive_value_check=True)
    return df


# ============================================================
# 3) Target detection + numeric coercion & impute
# ============================================================
TARGET_CANDIDATES = ["label","target","y","failure","bank_failure","default","is_failed"]

def detect_target_col(df: pd.DataFrame) -> str:
    for c in TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"Target column not found among {TARGET_CANDIDATES}")

def coerce_numeric_and_impute_with_train(df_tr, df_va, df_te, feat_cols: List[str]):
    feat_cols = sanitize_feature_list(feat_cols)

    for c in feat_cols:
        df_tr[c] = pd.to_numeric(df_tr[c], errors="coerce")
        df_va[c] = pd.to_numeric(df_va[c], errors="coerce")
        df_te[c] = pd.to_numeric(df_te[c], errors="coerce")

    df_tr[feat_cols] = df_tr[feat_cols].replace([np.inf, -np.inf], np.nan)
    df_va[feat_cols] = df_va[feat_cols].replace([np.inf, -np.inf], np.nan)
    df_te[feat_cols] = df_te[feat_cols].replace([np.inf, -np.inf], np.nan)

    med = df_tr[feat_cols].median(axis=0, skipna=True)
    df_tr[feat_cols] = df_tr[feat_cols].fillna(med)
    df_va[feat_cols] = df_va[feat_cols].fillna(med)
    df_te[feat_cols] = df_te[feat_cols].fillna(med)

    return df_tr, df_va, df_te

def make_xy(df_tr, df_va, df_te, target_col: str, feat_cols: List[str]):
    feat_cols = sanitize_feature_list(feat_cols)

    # STRICT: feature must exist in all splits
    missing = [c for c in feat_cols if (c not in df_tr.columns) or (c not in df_va.columns) or (c not in df_te.columns)]
    if missing:
        raise ValueError(
            "STRICT FEATURE MATCH FAILED.\n"
            f"Missing features in dataset. Example: {missing[:30]} (total {len(missing)})"
        )

    df_tr, df_va, df_te = coerce_numeric_and_impute_with_train(df_tr, df_va, df_te, feat_cols)

    X_tr = df_tr[feat_cols].to_numpy(np.float32)
    X_va = df_va[feat_cols].to_numpy(np.float32)
    X_te = df_te[feat_cols].to_numpy(np.float32)
    y_tr = df_tr[target_col].to_numpy(np.int64)
    y_va = df_va[target_col].to_numpy(np.int64)
    y_te = df_te[target_col].to_numpy(np.int64)
    return X_tr, y_tr, X_va, y_va, X_te, y_te


# ============================================================
# 4) Threshold (VAL best F1; match eval: n_grid=101)
# ============================================================
from sklearn.metrics import f1_score

def best_f1_threshold(y_true: np.ndarray, y_prob: np.ndarray, n_grid: int = 101) -> float:
    thresholds = np.linspace(0.0, 1.0, n_grid)
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)


# ============================================================
# 5) Deterministic training (CPU + single thread)
# ============================================================
def normalize_model_name(x: str) -> str:
    s = str(x).strip()
    m = {
        "XGB": "XGBoost",
        "XGBOOST": "XGBoost",
        "LGB": "LightGBM",
        "LGBM": "LightGBM",
        "LIGHTGBM": "LightGBM",
    }
    return m.get(s.upper(), s)

LGBM_DET_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=-1,
    deterministic=True,
    force_col_wise=True,
)

XGB_DET_PARAMS = dict(
    tree_method="hist",
    device="cpu",
    n_estimators=800,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0,
)

def train_model_and_threshold(model_name: str, X_tr, y_tr, X_va, y_va):
    model_name = normalize_model_name(model_name)

    if model_name == "LightGBM":
        clf = lgb.LGBMClassifier(**LGBM_DET_PARAMS)
        clf.fit(X_tr, y_tr)
        va_prob = clf.predict_proba(X_va)[:, 1]
        thr = best_f1_threshold(y_va, va_prob, n_grid=101)
        return clf, thr

    if model_name == "XGBoost":
        clf = xgb.XGBClassifier(**XGB_DET_PARAMS)
        clf.fit(X_tr, y_tr)
        va_prob = clf.predict_proba(X_va)[:, 1]
        thr = best_f1_threshold(y_va, va_prob, n_grid=101)
        return clf, thr

    raise ValueError(f"Only LightGBM/XGBoost supported. got={model_name}")


# ============================================================
# 6) SHAP/LIME helpers
# ============================================================
def shap_positive_class(shap_values):
    if isinstance(shap_values, (list, tuple)):
        if len(shap_values) == 2:
            return shap_values[1]
        return shap_values[-1]
    arr = np.array(shap_values)
    if arr.ndim == 3:
        return arr[:, :, -1]
    if arr.ndim == 2:
        return arr
    raise ValueError(f"Unsupported shap_values shape: {arr.shape}")

def expected_value_positive(explainer):
    ev = explainer.expected_value
    if isinstance(ev, (list, tuple, np.ndarray)) and np.array(ev).ndim >= 1 and len(ev) >= 2:
        return float(np.array(ev)[-1])
    return float(ev)

def pick_local_cases_with_labels(y_true, y_prob, thr, n_each: int = 2):
    pred = (y_prob >= thr).astype(int)
    TP = np.where((pred == 1) & (y_true == 1))[0]
    FP = np.where((pred == 1) & (y_true == 0))[0]
    FN = np.where((pred == 0) & (y_true == 1))[0]
    TN = np.where((pred == 0) & (y_true == 0))[0]

    def topk_high(idxs, score, k):
        if len(idxs) == 0: return []
        order = idxs[np.argsort(score[idxs])[::-1]]
        return order[:k].tolist()

    def topk_low(idxs, score, k):
        if len(idxs) == 0: return []
        order = idxs[np.argsort(score[idxs])]
        return order[:k].tolist()

    sel = []
    sel += [(i, "TP") for i in topk_high(TP, y_prob, n_each)]
    sel += [(i, "FP") for i in topk_high(FP, y_prob, n_each)]
    sel += [(i, "FN") for i in topk_low(FN,  y_prob, n_each)]
    sel += [(i, "TN") for i in topk_low(TN,  y_prob, n_each)]

    seen = set()
    chosen, labels = [], []
    for i, lab in sel:
        if int(i) not in seen:
            chosen.append(int(i))
            labels.append(lab)
            seen.add(int(i))
    return chosen, labels

def safe_tag(s: str) -> str:
    keep = []
    for ch in str(s):
        if ch.isalnum() or ch in {"_", "-", "."}:
            keep.append(ch)
        else:
            keep.append("_")
    return "".join(keep)

def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path

def out_dir_for(set_name: str) -> str:
    return ensure_dir(os.path.join(XAI_OUT_BASE, "GES", safe_tag(set_name)))

def run_shap_global_png(model, X_background, X_explain, feature_names, tag: str, out_dir: str,
                        max_background=2000, max_explain=5000):
    rng = np.random.RandomState(RANDOM_STATE)

    if X_background is not None and X_background.shape[0] > max_background:
        idx = rng.choice(X_background.shape[0], size=max_background, replace=False)
        Xb = X_background[idx]
    else:
        Xb = X_background

    if X_explain.shape[0] > max_explain:
        idx = rng.choice(X_explain.shape[0], size=max_explain, replace=False)
        Xe = X_explain[idx]
    else:
        Xe = X_explain

    explainer = shap.TreeExplainer(model, data=Xb, feature_perturbation="interventional")
    sv = explainer.shap_values(Xe, check_additivity=False)
    sv_pos = shap_positive_class(sv)

    plt.figure()
    shap.summary_plot(sv_pos, Xe, feature_names=feature_names, show=False)
    p1 = os.path.join(out_dir, f"shap_summary_{safe_tag(tag)}.png")
    plt.tight_layout()
    plt.savefig(p1, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] {p1}")

    plt.figure()
    shap.summary_plot(sv_pos, Xe, feature_names=feature_names, plot_type="bar", show=False)
    p2 = os.path.join(out_dir, f"shap_bar_{safe_tag(tag)}.png")
    plt.tight_layout()
    plt.savefig(p2, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] {p2}")

    return explainer

def save_shap_waterfall_png_for_case(explainer, shap_values_pos_full, X_test_row, idx_in_test: int,
                                     feature_names, tag: str, case_label: str,
                                     y_true_i: int, y_pred_i: int, y_prob_i: float, thr: float,
                                     out_dir: str):
    base = expected_value_positive(explainer)

    exp = shap.Explanation(
        values=shap_values_pos_full[idx_in_test],
        base_values=base,
        data=X_test_row,
        feature_names=feature_names,
    )

    plt.figure()
    shap.plots.waterfall(exp, show=False)

    title = f"{case_label} | idx={idx_in_test} | y={y_true_i} pred={y_pred_i} p={y_prob_i:.3f} thr={thr:.3f}"
    plt.title(title, fontsize=12, pad=12)

    fname = (
        f"shap_waterfall_{safe_tag(tag)}_{case_label}"
        f"_idx{idx_in_test}_y{y_true_i}_pred{y_pred_i}_p{y_prob_i:.3f}.png"
    )
    p = os.path.join(out_dir, fname)
    plt.tight_layout()
    plt.savefig(p, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"[SAVED] {p}")

def run_lime_png_for_cases(model, X_train, X_test, feature_names,
                           chosen_indices, chosen_labels,
                           y_true, y_prob, thr,
                           tag: str, out_dir: str, num_features=15):
    expl = LimeTabularExplainer(
        training_data=X_train,
        feature_names=feature_names,
        class_names=["ok", "fail"],
        mode="classification",
        discretize_continuous=True,
        random_state=RANDOM_STATE,
    )

    for idx, lab in zip(chosen_indices, chosen_labels):
        yt = int(y_true[idx])
        p = float(y_prob[idx])
        yp = int(p >= thr)

        exp = expl.explain_instance(
            data_row=X_test[idx],
            predict_fn=model.predict_proba,
            num_features=num_features
        )
        fig = exp.as_pyplot_figure()
        fig.suptitle(f"{lab} | idx={idx} | y={yt} pred={yp} p={p:.3f} thr={thr:.3f}",
                     fontsize=12, y=1.02)

        fname = f"lime_{safe_tag(tag)}_{lab}_idx{idx}_y{yt}_pred{yp}_p{p:.3f}.png"
        outp = os.path.join(out_dir, fname)
        fig.tight_layout()
        fig.savefig(outp, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"[SAVED] {outp}")


# ============================================================
# 7) Optional metadata
# ============================================================
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def get_versions() -> Dict[str, str]:
    import sklearn as _sk
    return {
        "python": f"{os.sys.version_info.major}.{os.sys.version_info.minor}.{os.sys.version_info.micro}",
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": _sk.__version__,
        "xgboost": xgb.__version__,
        "lightgbm": lgb.__version__,
        "shap": shap.__version__,
    }

def maybe_save_metadata(out_dir: str, meta: Dict[str, Any]):
    if not SAVE_METADATA:
        return
    p = os.path.join(out_dir, "run_metadata.json")
    with open(p, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"[META] saved -> {p}")


# ============================================================
# 8) Infer experiment settings + STRICT feature retrieval
# ============================================================
def infer_int_from_filename(path: str, pattern: str) -> int:
    m = re.search(pattern, os.path.basename(path))
    if not m:
        raise ValueError(f"Cannot infer integer with pattern={pattern} from filename={os.path.basename(path)}")
    return int(m.group(1))

def count_original_features(feature_list_path: str) -> int:
    fl = read_csv_safely(feature_list_path, "feature_list")
    if "GROUP" in fl.columns:
        return int((fl["GROUP"].astype(str).str.lower().str.strip() == "original").sum())
    if "FEATURE_TYPE" in fl.columns:
        return int((fl["FEATURE_TYPE"].astype(str).str.lower().str.strip() == "original").sum())
    if "FEATURE_NAME" in fl.columns:
        names = fl["FEATURE_NAME"].astype(str).tolist()
        return int(sum(1 for n in names if not n.lower().startswith("edge_")))
    raise ValueError("Cannot determine number of original features from feature_list.csv")

def pick_feature_key_strict(features_used_df: pd.DataFrame, set_name: str, dag: str, k_edge: int, n_feat: int) -> str:
    need = {"FEATURE_KEY","SET","DAG","K_EDGE","N_FEAT","FEATURES_JSON"}
    miss = [c for c in need if c not in features_used_df.columns]
    if miss:
        raise ValueError(f"features_used.csv missing columns: {miss}")

    cand = features_used_df.copy()
    cand["SET"] = cand["SET"].astype(str).str.strip()
    cand["DAG"] = cand["DAG"].astype(str).str.strip()

    def to_int(x):
        try:
            return int(float(x))
        except Exception:
            return None

    cand["K_EDGE_INT"] = cand["K_EDGE"].apply(to_int)
    cand["N_FEAT_INT"] = cand["N_FEAT"].apply(to_int)

    sub = cand[
        (cand["SET"] == set_name) &
        (cand["DAG"] == dag) &
        (cand["K_EDGE_INT"] == int(k_edge)) &
        (cand["N_FEAT_INT"] == int(n_feat))
    ].copy()

    if sub.empty:
        raise ValueError(
            "No matching row in features_used.csv for "
            f"SET={set_name}, DAG={dag}, K_EDGE={k_edge}, N_FEAT={n_feat}"
        )

    uniq = sub["FEATURES_JSON"].astype(str).nunique()
    if uniq != 1:
        ex = sub[["FEATURE_KEY","SEED","TRIAL","N_FEAT","K_EDGE"]].head(10)
        raise ValueError(
            "Ambiguous FEATURES_JSON: multiple different feature lists match the same condition.\n"
            f"SET={set_name}, DAG={dag}, K_EDGE={k_edge}, N_FEAT={n_feat}\n"
            f"Matched rows={len(sub)} FEATURES_JSON unique={uniq}\n"
            f"Example rows:\n{ex.to_string(index=False)}"
        )

    return str(sub["FEATURE_KEY"].iloc[0]).strip()

def load_features_json(features_used_df: pd.DataFrame, feature_key: str) -> List[str]:
    row = features_used_df.loc[features_used_df["FEATURE_KEY"].astype(str).str.strip() == str(feature_key).strip()]
    if row.empty:
        raise ValueError(f"FEATURE_KEY not found: {feature_key}")
    cols = json.loads(row["FEATURES_JSON"].iloc[0])
    cols = sanitize_feature_list(cols)
    if len(cols) == 0:
        raise ValueError(f"All features removed after sanitization. FEATURE_KEY={feature_key}")
    return cols


# ============================================================
# 9) Main runner
# ============================================================
def load_splits():
    df_tr = read_csv_safely(DATA_TRAIN, "GES_train")
    df_va = read_csv_safely(DATA_VAL,   "GES_val")
    df_te = read_csv_safely(DATA_TEST,  "GES_test")
    return df_tr, df_va, df_te

def run_one_xai(set_name: str, model_name: str, feat_cols: List[str],
                n_each_case: int = 2, lime_num_features: int = 15,
                max_background: int = 2000, max_explain: int = 5000):

    set_seed_everywhere(RANDOM_STATE)
    model_name = normalize_model_name(model_name)

    out_dir = out_dir_for(set_name)
    df_tr, df_va, df_te = load_splits()
    target_col = detect_target_col(df_tr)

    feat_cols = sanitize_feature_list(feat_cols)
    X_tr, y_tr, X_va, y_va, X_te, y_te = make_xy(df_tr, df_va, df_te, target_col, feat_cols)

    model, thr = train_model_and_threshold(model_name, X_tr, y_tr, X_va, y_va)
    te_prob = model.predict_proba(X_te)[:, 1]
    te_pred = (te_prob >= thr).astype(int)

    tag = f"{set_name}_GES_{model_name}_NFEAT{len(feat_cols)}"
    print(f"\n==== XAI TARGET ====")
    print(f"SET={set_name} DAG=GES MODEL={model_name} n_feat={len(feat_cols)} thr={thr:.3f}")
    print(f"[OUT] {out_dir}")

    # SHAP global
    run_shap_global_png(
        model=model,
        X_background=X_tr,
        X_explain=X_te,
        feature_names=feat_cols,
        tag=tag,
        out_dir=out_dir,
        max_background=max_background,
        max_explain=max_explain
    )

    # SHAP full test for waterfall
    rng = np.random.RandomState(RANDOM_STATE)
    Xb = X_tr
    if Xb.shape[0] > max_background:
        Xb = Xb[rng.choice(Xb.shape[0], size=max_background, replace=False)]

    explainer_full = shap.TreeExplainer(model, data=Xb, feature_perturbation="interventional")
    sv_full = explainer_full.shap_values(X_te, check_additivity=False)
    sv_full_pos = shap_positive_class(sv_full)

    chosen, chosen_labels = pick_local_cases_with_labels(y_te, te_prob, thr, n_each=n_each_case)
    print("Chosen indices:", chosen)
    print("Chosen labels:", chosen_labels)

    for idx, lab in zip(chosen, chosen_labels):
        yt = int(y_te[idx])
        yp = int(te_pred[idx])
        p = float(te_prob[idx])
        save_shap_waterfall_png_for_case(
            explainer=explainer_full,
            shap_values_pos_full=sv_full_pos,
            X_test_row=X_te[idx],
            idx_in_test=idx,
            feature_names=feat_cols,
            tag=tag,
            case_label=lab,
            y_true_i=yt,
            y_pred_i=yp,
            y_prob_i=p,
            thr=thr,
            out_dir=out_dir
        )

    # LIME per-case
    run_lime_png_for_cases(
        model=model,
        X_train=X_tr,
        X_test=X_te,
        feature_names=feat_cols,
        chosen_indices=chosen,
        chosen_labels=chosen_labels,
        y_true=y_te,
        y_prob=te_prob,
        thr=thr,
        tag=tag,
        out_dir=out_dir,
        num_features=lime_num_features
    )

    meta = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "seed": int(RANDOM_STATE),
        "set": set_name,
        "dag": "GES",
        "model": model_name,
        "n_feat": int(len(feat_cols)),
        "threshold_val_best_f1": float(thr),
        "versions": get_versions(),
        "params": {
            "LightGBM": LGBM_DET_PARAMS if model_name == "LightGBM" else None,
            "XGBoost": XGB_DET_PARAMS if model_name == "XGBoost" else None,
        },
        "files_sha256": {
            "features_used.csv": sha256_file(FEATURES_USED_PATH),
            "feature_list.csv": sha256_file(FEATURE_LIST_PATH),
            "data_GES_train.csv": sha256_file(DATA_TRAIN),
            "data_GES_val.csv": sha256_file(DATA_VAL),
            "data_GES_test.csv": sha256_file(DATA_TEST),
            "results_F_mean.csv": sha256_file(RESULTS_F_MEAN_PATH),
            "results_OF_mean.csv": sha256_file(RESULTS_OF_MEAN_PATH),
        },
        "features_head": feat_cols[:50],
    }
    maybe_save_metadata(out_dir, meta)


# -------------------------
# Run: infer settings + load exact features from features_used.csv
# -------------------------
n_orig = count_original_features(FEATURE_LIST_PATH)

F_K_EDGE = infer_int_from_filename(RESULTS_F_MEAN_PATH, r"_(\d+)edges_")
F_N_FEAT = F_K_EDGE

OF_N_FEAT = infer_int_from_filename(RESULTS_OF_MEAN_PATH, r"_(\d+)features_")
OF_K_EDGE = OF_N_FEAT - n_orig
if OF_K_EDGE <= 0:
    raise ValueError(f"Invalid inferred OF_K_EDGE={OF_K_EDGE}. Check n_orig={n_orig} and OF_N_FEAT={OF_N_FEAT}.")

print("\n[INFER]")
print("n_orig =", n_orig)
print("F : K_EDGE =", F_K_EDGE, "N_FEAT =", F_N_FEAT)
print("OF: K_EDGE =", OF_K_EDGE, "N_FEAT =", OF_N_FEAT)

feat_used = pd.read_csv(FEATURES_USED_PATH)

F_feature_key  = pick_feature_key_strict(feat_used, set_name="F",  dag="GES", k_edge=F_K_EDGE,  n_feat=F_N_FEAT)
OF_feature_key = pick_feature_key_strict(feat_used, set_name="OF", dag="GES", k_edge=OF_K_EDGE, n_feat=OF_N_FEAT)

F_cols  = load_features_json(feat_used, F_feature_key)
OF_cols = load_features_json(feat_used, OF_feature_key)

print("\n[FEATURE_KEY]")
print("F :",  F_feature_key,  "n_feat =", len(F_cols))
print("OF:",  OF_feature_key, "n_feat =", len(OF_cols))

# decide model
dfF_mean  = pd.read_csv(RESULTS_F_MEAN_PATH)
dfOF_mean = pd.read_csv(RESULTS_OF_MEAN_PATH)

model_F  = DEFAULT_MODEL_NAME
model_OF = DEFAULT_MODEL_NAME
if "MODEL" in dfF_mean.columns:
    model_F = str(dfF_mean["MODEL"].iloc[0])
if "MODEL" in dfOF_mean.columns:
    model_OF = str(dfOF_mean["MODEL"].iloc[0])

print("\n[MODEL]")
print("F :", model_F)
print("OF:", model_OF)

# Run XAI
run_one_xai("F",  model_F,  F_cols,  n_each_case=2, lime_num_features=15)
run_one_xai("OF", model_OF, OF_cols, n_each_case=2, lime_num_features=15)

print("\n[DONE] outputs saved to:", XAI_OUT_BASE)


[CWD] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight
[OK] all paths exist

[INFER]
n_orig = 13
F : K_EDGE = 34 N_FEAT = 34
OF: K_EDGE = 34 N_FEAT = 47

[FEATURE_KEY]
F : b154670abb75bb96 n_feat = 34
OF: 084da68000a6f7f1 n_feat = 47

[MODEL]
F : LightGBM
OF: LightGBM

==== XAI TARGET ====
SET=F DAG=GES MODEL=LightGBM n_feat=34 thr=0.060
[OUT] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\F


100%|===================| 3573/3577 [04:01<00:00]        

[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\F\shap_summary_F_GES_LightGBM_NFEAT34.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\F\shap_bar_F_GES_LightGBM_NFEAT34.png


100%|===================| 3563/3577 [04:00<00:00]        

Chosen indices: [2372, 2692, 3157, 2256, 2870, 2375, 3330, 600]
Chosen labels: ['TP', 'TP', 'FP', 'FP', 'FN', 'FN', 'TN', 'TN']
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\F\shap_waterfall_F_GES_LightGBM_NFEAT34_TP_idx2372_y1_pred1_p1.000.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\F\shap_waterfall_F_GES_LightGBM_NFEAT34_TP_idx2692_y1_pred1_p1.000.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\F\shap_waterfall_F_GES_LightGBM_NFEAT34_FP_idx3157_y0_pred1_p0.996.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\F\shap_waterfall_F_GES_LightGBM_NFEAT34_FP_idx2256_y0_pred1_p0.994.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\F\shap_waterfall_F_GES_LightGBM_NFEAT34_FN_idx28

100%|===================| 3571/3577 [04:11<00:00]        

[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\OF\shap_summary_OF_GES_LightGBM_NFEAT47.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\OF\shap_bar_OF_GES_LightGBM_NFEAT47.png


100%|===================| 3563/3577 [04:18<00:01]        

Chosen indices: [2372, 671, 2471, 639, 2870, 354, 3187, 3168]
Chosen labels: ['TP', 'TP', 'FP', 'FP', 'FN', 'FN', 'TN', 'TN']
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\OF\shap_waterfall_OF_GES_LightGBM_NFEAT47_TP_idx2372_y1_pred1_p1.000.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\OF\shap_waterfall_OF_GES_LightGBM_NFEAT47_TP_idx671_y1_pred1_p1.000.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\OF\shap_waterfall_OF_GES_LightGBM_NFEAT47_FP_idx2471_y0_pred1_p0.999.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\OF\shap_waterfall_OF_GES_LightGBM_NFEAT47_FP_idx639_y0_pred1_p0.998.png
[SAVED] d:\University\3-2.5\PADA_Lab\Bank_Failure_Prediction_3\9_best_model_no_weight\xai_outputs_repro\GES\OF\shap_waterfall_OF_GES_LightGBM_NFEAT47_FN